<a href="https://colab.research.google.com/github/siddikiranmayi-cloud/Neural_Network_SMS_Text-Classifier-/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np

# Use standard TextVectorization layer
from tensorflow.keras.layers import TextVectorization, Embedding, Dense, GlobalAveragePooling1D

print("TensorFlow version:", tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load TSV files (they have no headers, column 0 is label, column 1 is text)
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

# Convert labels ('ham' -> 0, 'spam' -> 1)
train_labels = train_df['label'].map({'ham': 0, 'spam': 1}).values
test_labels = test_df['label'].map({'ham': 0, 'spam': 1}).values

train_sentences = train_df['message'].values
test_sentences = test_df['message'].values

In [ ]:

# Define Text Hyperparameters
VOCAB_SIZE = 10000
MAX_LEN = 250
EMBEDDING_DIM = 64  # Increased embedding size for better pattern capture

# Create text vectorization mapping
vectorizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorizer.adapt(train_sentences)

# Build Sequential Model with a Bidirectional LSTM
model = tf.keras.Sequential([
    vectorizer,
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)), # Context-aware sequence processing
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Train the upgraded model
model.fit(
    train_sentences,
    train_labels,
    epochs=10,
    validation_data=(test_sentences, test_labels),
    verbose=1
)

In [ ]:

# function to predict messages based on model
def predict_message(pred_text):
  input_tensor = tf.constant([pred_text])
  probability = model.predict(input_tensor, verbose=0)[0][0]
  label = "spam" if probability >= 0.5 else "ham"
  return [float(probability), label]

pred_text = "how are you doing today?"
print(predict_message(pred_text))

In [ ]:

# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()